# Notebook 02: Concurrencia, Asincronía y asyncio

**Módulo 16 — Clase 2**

---

In [ ]:
import asyncio
import time
import threading
import os
import sys

print(f'Python {sys.version}')
print(f'asyncio version: {asyncio.__version__ if hasattr(asyncio, "__version__") else "built-in"}')

## Sección 1: await secuencial vs asyncio.gather

In [ ]:
async def tarea_io(nombre: str, duracion: float) -> str:
    inicio = time.perf_counter()
    await asyncio.sleep(duracion)
    elapsed = time.perf_counter() - inicio
    return f'{nombre}: {elapsed:.2f}s'

DURACION = 1.0
N_TAREAS = 5

t0 = time.perf_counter()
resultados_m2 = []
for i in range(N_TAREAS):
    r = await tarea_io(f'τ{i+1}', DURACION)
    resultados_m2.append(r)
t_m2 = time.perf_counter() - t0

print(f'=== M2: await secuencial ===')
for r in resultados_m2:
    print(f'  {r}')
print(f'Tiempo total M2: {t_m2:.2f}s  (esperado: {N_TAREAS * DURACION:.1f}s = N×T)')
print()

In [ ]:
t0 = time.perf_counter()
resultados_m4 = await asyncio.gather(
    *[tarea_io(f'τ{i+1}', DURACION) for i in range(N_TAREAS)]
)
t_m4 = time.perf_counter() - t0

print(f'=== M4: asyncio.gather ===')
for r in resultados_m4:
    print(f'  {r}')
print(f'Tiempo total M4: {t_m4:.2f}s  (esperado: ~{DURACION:.1f}s = T_max)')
print()
print(f'Speedup M4/M2: {t_m2/t_m4:.1f}x')
print(f'Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅')
print(f'Las {N_TAREAS} tareas de {DURACION}s corren en ~{DURACION}s en lugar de {N_TAREAS*DURACION}s')

## Sección 2: asyncio.sleep vs time.sleep — event loop bloqueado

In [ ]:
loop = asyncio.get_event_loop()
loop.set_debug(True)
loop.slow_callback_duration = 0.05

async def tarea_correcta(nombre: str):
    print(f'  {nombre}: inicio')
    await asyncio.sleep(0.3)
    print(f'  {nombre}: fin')

async def tarea_bloqueante(nombre: str):
    print(f'  {nombre}: inicio')
    time.sleep(0.3)
    print(f'  {nombre}: fin')

print('=== gather con tareas CORRECTAS (asyncio.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_correcta('A'), tarea_correcta('B'), tarea_correcta('C'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.3s)\n')

print('=== gather con tareas BLOQUEANTES (time.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_bloqueante('X'), tarea_bloqueante('Y'), tarea_bloqueante('Z'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.9s — sin mejora)')
print()
print('Observa: con time.sleep, gather NO ayuda.')
print('time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.')

In [ ]:
loop.set_debug(False)

## Sección 3: M3 threading CPU-bound — confirmando el GIL

In [ ]:
def tarea_cpu_bound(n: int) -> int:
    return sum(range(n))

N_CPU = 30_000_000
N_HILOS = 4

t0 = time.perf_counter()
for _ in range(N_HILOS):
    tarea_cpu_bound(N_CPU)
t_secuencial = time.perf_counter() - t0

t0 = time.perf_counter()
hilos = [threading.Thread(target=tarea_cpu_bound, args=(N_CPU,)) for _ in range(N_HILOS)]
for h in hilos: h.start()
for h in hilos: h.join()
t_threading = time.perf_counter() - t0

print(f'M1 secuencial ({N_HILOS} tareas): {t_secuencial:.2f}s')
print(f'M3 threading  ({N_HILOS} hilos):  {t_threading:.2f}s')
print(f'Speedup: {t_secuencial/t_threading:.2f}x  (esperado ~{N_HILOS}x, real ~1x por el GIL)')
print()
print('M2 = await secuencial: idéntico a M1 porque no hay overlap entre coroutines.')
print('La condición que falta en M2: exec(τⱼ) ∩ wait(τᵢ) ≠ ∅ → gather la satisface, el for no.')
print('M3 = threading CPU-bound: el GIL nunca se libera → sin speedup, incluso más lento.')

## Sección 4: Race condition + fix con Lock

In [ ]:
N_INCREMENTOS = 100_000
N_HILOS_RACE = 4

contador_sin_lock = [0]

def incrementar_sin_lock():
    for _ in range(N_INCREMENTOS):
        contador_sin_lock[0] += 1

hilos = [threading.Thread(target=incrementar_sin_lock) for _ in range(N_HILOS_RACE)]
for h in hilos: h.start()
for h in hilos: h.join()

esperado = N_INCREMENTOS * N_HILOS_RACE
print(f'Sin lock  — esperado: {esperado:,}, obtenido: {contador_sin_lock[0]:,}')
print(f'Diferencia: {esperado - contador_sin_lock[0]:,} incrementos perdidos')
print()

lock = threading.Lock()
contador_con_lock = [0]

def incrementar_con_lock():
    for _ in range(N_INCREMENTOS):
        with lock:
            contador_con_lock[0] += 1

hilos = [threading.Thread(target=incrementar_con_lock) for _ in range(N_HILOS_RACE)]
for h in hilos: h.start()
for h in hilos: h.join()

print(f'Con lock  — esperado: {esperado:,}, obtenido: {contador_con_lock[0]:,}')
assert contador_con_lock[0] == esperado, 'Error: el lock no funcionó'
print('✓ El lock garantiza atomicidad — no se pierden incrementos.')

## Sección 5: Chatbot v2 con asyncio — N usuarios concurrentes

In [ ]:
import random

async def consultar_bd(user_id: int) -> list:
    await asyncio.sleep(0.05)
    return [f'historial de usuario {user_id}']

async def llamar_llm(historial: list) -> str:
    await asyncio.sleep(random.uniform(1.0, 2.0))
    return f'respuesta para: {historial[-1]}'

async def handle_request(user_id: int) -> dict:
    t_inicio = time.perf_counter()
    historial = await consultar_bd(user_id)
    respuesta = await llamar_llm(historial)
    latencia = time.perf_counter() - t_inicio
    return {'user': user_id, 'respuesta': respuesta, 'latencia': latencia}

async def servidor_v1(n_usuarios: int):
    resultados = []
    for i in range(n_usuarios):
        r = await handle_request(i)
        resultados.append(r)
    return resultados

async def servidor_v2(n_usuarios: int):
    resultados = await asyncio.gather(*[handle_request(i) for i in range(n_usuarios)])
    return resultados

N = 10
print(f'Comparando v1 (secuencial) vs v2 (concurrent) con {N} usuarios...')
print()

t0 = time.perf_counter()
res_v1 = await servidor_v1(N)
t_v1 = time.perf_counter() - t0
lat_v1 = sum(r['latencia'] for r in res_v1) / N

t0 = time.perf_counter()
res_v2 = await servidor_v2(N)
t_v2 = time.perf_counter() - t0
lat_v2 = sum(r['latencia'] for r in res_v2) / N

print(f'v1 secuencial: {t_v1:.2f}s totales, latencia promedio {lat_v1:.2f}s/usuario')
print(f'v2 concurrent: {t_v2:.2f}s totales, latencia promedio {lat_v2:.2f}s/usuario')
print(f'Speedup: {t_v1/t_v2:.1f}x más rápido')
print()
print(f'Cada usuario en v2 tiene latencia similar (~max del LLM) porque sus esperas se solapan.')
print(f'En v1 el usuario {N} espera la suma de las {N} latencias anteriores.')